# ChessPublishing Atom Extraction Visualizer

Browse extracted atoms from `data/extraction_included.jsonl`.  
Each entry shows: board position, played move, original annotation, and extracted reasoning atoms.

In [5]:
import json
import random
from pathlib import Path
from collections import Counter

import chess
import chess.svg
from IPython.display import SVG, display, HTML, Markdown, clear_output

DATA_PATH = Path('data/chesspublishinga_included.jsonl')

with open(DATA_PATH) as f:
    entries = [json.loads(line) for line in f]

print(f'Loaded {len(entries)} entries from {DATA_PATH.name}')

Loaded 398 entries from chesspublishinga_included.jsonl


---
## 1. Overview Statistics

In [6]:
mainline = sum(1 for e in entries if e['is_mainline'])
variation = len(entries) - mainline

# Atom counts
atom_counts = [len(e['extracted'].get('reasoning', [])) for e in entries]
total_atoms = sum(atom_counts)
atom_dist = Counter(atom_counts)

# Annotation lengths
ann_lengths = [len(e['annotation']) for e in entries]

# Games
games = Counter(e['game'] for e in entries)

# Entries with variations in extracted
has_variation = sum(1 for e in entries if e['extracted'].get('variation'))

# Entries with alternatives
has_alternative = sum(1 for e in entries if e['extracted'].get('alternative'))

lines = [
    f'**Total entries:** {len(entries)}',
    f'**Mainline:** {mainline} | **Variation:** {variation}',
    '',
    f'**Total atoms:** {total_atoms}',
    f'**Atoms per entry:** mean={total_atoms/len(entries):.1f}, '
    f'median={sorted(atom_counts)[len(atom_counts)//2]}, '
    f'min={min(atom_counts)}, max={max(atom_counts)}',
    '',
    '**Atom count distribution:**',
]
for n in sorted(atom_dist):
    lines.append(f'- {n} atoms: {atom_dist[n]} entries')

lines.append('')
lines.append(f'**Entries with extracted variation line:** {has_variation}')
lines.append(f'**Entries with extracted alternative(s):** {has_alternative}')
lines.append('')
lines.append(f'**Annotation length:** mean={sum(ann_lengths)/len(ann_lengths):.0f}, '
             f'median={sorted(ann_lengths)[len(ann_lengths)//2]}, '
             f'max={max(ann_lengths)}')
lines.append('')
lines.append(f'**Unique games:** {len(games)}')
lines.append('')
lines.append('**Top 10 games by entry count:**')
for g, cnt in games.most_common(10):
    lines.append(f'- {g}: {cnt}')

display(Markdown('\n'.join(lines)))

**Total entries:** 398
**Mainline:** 151 | **Variation:** 247

**Total atoms:** 641
**Atoms per entry:** mean=1.6, median=1, min=1, max=8

**Atom count distribution:**
- 1 atoms: 220 entries
- 2 atoms: 138 entries
- 3 atoms: 26 entries
- 4 atoms: 8 entries
- 5 atoms: 4 entries
- 7 atoms: 1 entries
- 8 atoms: 1 entries

**Entries with extracted variation line:** 154
**Entries with extracted alternative(s):** 60

**Annotation length:** mean=135, median=100, max=909

**Unique games:** 381

**Top 10 games by entry count:**
- Muniz, Francisco Allysson vs Williams, Simon Kim: 3
- Andersen, Mads vs Arvola, B.: 2
- Maghsoodloo, Parham vs Karjakin, Sergey: 2
- Kosten, Anthony C vs Grimberg: 2
- Rusev, Krasimir vs Brynell, Stellan: 2
- Carlsen, Magnus vs Vachier Lagrave, Maxime: 2
- Ponomariov, Ruslan vs Motylev, Alexander: 2
- Duda, Jan Krzysztof vs Carlsen, Magnus: 2
- Caruana, Fabiano vs Ding, Liren: 2
- Duda, Jan-Krzysztof vs Xiong, Jeffery: 2

---
## 2. Browse Entries

Board + move + original annotation + extracted atoms.

In [7]:
import html as html_mod

def render_entry(entry, idx, total, size=320):
    """Render one entry as HTML with board + annotation + full structured JSON."""
    board = chess.Board(entry['fen'])
    extracted = entry['extracted']

    # Highlight the move
    arrows = []
    if entry.get('move_uci'):
        move = chess.Move.from_uci(entry['move_uci'])
        arrows = [chess.svg.Arrow(move.from_square, move.to_square, color='#88cc44aa')]

    svg = chess.svg.board(board, arrows=arrows, size=size)
    turn = 'White' if board.turn else 'Black'
    move_san = entry.get('move_san') or '(none)'
    loc = 'mainline' if entry['is_mainline'] else 'variation'

    # Parent comment if present
    parent_html = ''
    if entry.get('parent_comment'):
        parent_html = f"""<div style='margin-top:4px; color:#666; font-style:italic'>
            <b>Context:</b> {entry['parent_comment']}</div>"""

    ann = entry['annotation']
    ann_display = ann[:700] + ('...' if len(ann) > 700 else '')

    # Full structured JSON of the extracted field
    extracted_json = html_mod.escape(json.dumps(extracted, indent=2))

    return f"""
    <div style='display:flex; gap:24px; align-items:flex-start; margin:12px 0;
                padding:12px; border:1px solid #ddd; border-radius:8px; background:#fafafa'>
        <div style='flex-shrink:0'>{svg}</div>
        <div style='font-family:monospace; font-size:13px; line-height:1.7; min-width:0'>
            <div style='font-size:16px; font-weight:bold; color:#2255aa'>
                {entry['game']}
            </div>
            <div><b>Entry:</b> {idx+1} / {total} &nbsp; (<code>{entry.get('custom_id', '')}</code>)</div>
            <div><b>Move:</b> {move_san} ({turn} to move) &nbsp; | &nbsp; <b>Location:</b> {loc}</div>
            {parent_html}
            <div style='margin-top:8px; white-space:pre-wrap; font-family:serif; font-size:14px;
                        border-left:3px solid #888; padding-left:10px'>
                <b>Original annotation:</b><br>{ann_display}
            </div>
            <div style='margin-top:10px'>
                <b>Extracted:</b>
                <pre style='background:#f0f0f0; padding:8px; border-radius:4px;
                            font-size:12px; line-height:1.4; overflow-x:auto;
                            white-space:pre-wrap; word-break:break-word'>{extracted_json}</pre>
            </div>
            <div style='margin-top:4px; font-size:10px; color:#999; word-break:break-all'>
                FEN: {entry['fen']}
            </div>
        </div>
    </div>
    """


def browse(filtered, label='entries'):
    """Interactive browser. Enter = next, q = stop."""
    if not filtered:
        print(f'No {label} found.')
        return
    print(f'{len(filtered)} {label}. Press Enter for next, q to stop.')
    for i, entry in enumerate(filtered):
        clear_output(wait=True)
        display(HTML(render_entry(entry, i, len(filtered))))
        try:
            resp = input(f'[{i+1}/{len(filtered)}] Enter=next, q=stop: ')
            if resp.strip().lower() in ('q', 'quit', 'exit'):
                break
        except (KeyboardInterrupt, EOFError):
            break
    print(f'Stopped at {i+1}/{len(filtered)}')


print('Helpers ready.')

Helpers ready.


---
## 3. Static Samples

5 random examples from each atom-count bucket.

In [8]:
random.seed(42)

buckets = [
    ('3+ atoms', [e for e in entries if len(e['extracted'].get('reasoning', [])) >= 3], '#00aa00'),
    ('2 atoms',  [e for e in entries if len(e['extracted'].get('reasoning', [])) == 2], '#cc8800'),
    ('1 atom',   [e for e in entries if len(e['extracted'].get('reasoning', [])) == 1], '#cc3333'),
]

for label, subset, color in buckets:
    sample = random.sample(subset, min(5, len(subset)))
    html_parts = [f'<h2 style="color:{color}">{label} ({len(subset)} total)</h2>']
    for i, e in enumerate(sample):
        html_parts.append(render_entry(e, i, len(sample), size=260))
    display(HTML(''.join(html_parts)))

---
## 4. Browse All Entries

In [9]:
browse(entries, 'all entries')

Stopped at 2/398


---
## 5. Entries with Alternatives or Variation Lines

In [10]:
alt_or_var = [e for e in entries
              if e['extracted'].get('variation') or e['extracted'].get('alternative')]
browse(alt_or_var, 'entries with alternatives or variation lines')

Stopped at 2/181


---
## 6. Custom Filter

Edit the filter below to browse specific slices.

In [11]:
filtered = [
    e for e in entries
    if len(e['extracted'].get('reasoning', [])) >= 3
    # and e['is_mainline']
    # and 'sacrifice' in e['annotation'].lower()
]
browse(filtered, 'custom filter')

Stopped at 1/40
